# LLM Factor Mining Agent

QuantaAlpha 式闭环：多方向假设 → 因子生成（硬约束） → IS 网格搜索 → 横向评审 → OOS 验证 → 研究轨迹进化。

主循环：`one_batch()` — 一个 batch 完成一次完整的研究迭代。

In [1]:
import requests
import json
import os
import re
import sys
import time

# Ensure local modules are discoverable
sys.path.insert(0, ".")

def load_env(path=".env"):
    """Load KEY=VALUE pairs from .env file (gitignored)."""
    if os.path.exists(path):
        with open(path, encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line or line.startswith("#") or "=" not in line:
                    continue
                k, v = line.split("=", 1)
                os.environ.setdefault(k.strip(), v.strip())

load_env()

API_KEY = os.environ.get("OPENCODE_GO_API_KEY", "")
if not API_KEY:
    raise RuntimeError("Missing OPENCODE_GO_API_KEY: set it in .env or environment")
BASE_URL = "https://opencode.ai/zen/go/v1"

HEADERS = {
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json",
}

MODEL = "deepseek-v4-pro"
print(f"Model: {MODEL}")

Model: deepseek-v4-pro


In [2]:
def chat(messages, model=None, max_tokens=8192, temperature=0.8):
    """Call OpenCode Go chat/completions API with retry on transient errors."""
    if model is None:
        model = MODEL

    data = {
        "model": model,
        "messages": messages,
        "max_tokens": max_tokens,
        "temperature": temperature,
    }

    retry_status = {429, 500, 502, 503, 504}
    max_retries = 5
    backoff = [2, 4, 8, 16, 32]

    for attempt in range(max_retries + 1):
        try:
            resp = requests.post(f"{BASE_URL}/chat/completions", headers=HEADERS, json=data, timeout=300)

            if resp.status_code == 200:
                result = resp.json()
                msg = result["choices"][0]["message"]
                content = msg.get("content", "") or ""
                reasoning = msg.get("reasoning_content", "") or ""
                usage = result.get("usage", {})

                return {
                    "content": content,
                    "reasoning": reasoning,
                    "answer": content if content.strip() else reasoning,
                    "usage": usage,
                    "model": result.get("model", ""),
                }

            if resp.status_code in retry_status and attempt < max_retries:
                wait = backoff[attempt]
                print(f"[retry] {resp.status_code} -> sleep {wait}s (attempt {attempt + 1}/{max_retries})")
                time.sleep(wait)
                continue

            raise RuntimeError(f"API error {resp.status_code}: {resp.text[:300]}")

        except (requests.exceptions.ConnectionError, requests.exceptions.Timeout) as e:
            if attempt < max_retries:
                wait = backoff[attempt]
                print(f"[retry] {type(e).__name__} -> sleep {wait}s (attempt {attempt + 1}/{max_retries})")
                time.sleep(wait)
                continue
            raise RuntimeError(f"API connection error: {e}")

    raise RuntimeError(f"API error: retries exhausted after {max_retries} attempts")


def quick_test():
    """Verify API connectivity."""
    resp = requests.get(f"{BASE_URL}/models", headers=HEADERS)
    if resp.status_code == 200:
        models = [m["id"] for m in resp.json().get("data", [])]
        print(f"[OK] {len(models)} models available: {models[:5]}...")
        result = chat([{"role": "user", "content": "Say hi."}], max_tokens=10)
        print(f"[OK] Tokens: in={result['usage'].get('prompt_tokens')}, out={result['usage'].get('completion_tokens')}")
    else:
        print(f"[FAIL] Models endpoint: {resp.status_code}")

quick_test()

[OK] 26 models available: ['minimax-m3', 'minimax-m2.7', 'minimax-m2.5', 'kimi-k3', 'kimi-k2.7-code']...
[OK] Tokens: in=86, out=10


## 初始化

回测引擎 + 研究轨迹（跨 batch 记忆）。

In [3]:
from backtest.engine import FactorBacktester
from agent.trajectory import ResearchTrajectory

# 先跑 BTCUSDT，后续扩展多币种
bt = FactorBacktester("data/SANDUSDT_1H.csv", commission_bps=6)
trajectory = ResearchTrajectory("trajectory.json")

print(f"Symbol: {bt.symbol}")
print(f"IS:     {bt.is_range} ({bt.is_rows} rows)")
print(f"OOS:    {bt.oos_range} ({bt.oos_rows} rows)")
print(f"Commission: {bt.commission_bps} bps")
print(f"轨迹方向数: {len(trajectory.data['directions'])}")

Symbol: SANDUSDT_1H
IS:     ('2021-01-25', '2025-06-29') (38778 rows)
OOS:    ('2025-06-29', '2026-06-29') (8760 rows)
Commission: 6 bps
轨迹方向数: 0


## Agent Batch 主循环

一个 batch：方向假设 → 因子生成 → IS网格搜索+代码筛选 → OOS → 轨迹更新。

筛选阈值在 `config.json`，改动后重新执行本 cell 生效。

In [4]:
from agent.hypothesis import generate_directions
from agent.factor_gen import generate_factor as gen_factor
from agent.judge import ask_oos_judge
from agent.config import load_config
from agent.screener import screen_factor

# 筛选阈值（改 config.json 即可调整）
cfg = load_config("config.json")


def one_batch(max_directions: int = 6) -> dict:
    """One full research iteration."""
    print("=" * 70)
    print("BATCH START")
    print(f"[筛选配置] {cfg}")

    # ---- Step 1: Hypothesis Agent ----
    print("\n[1] Hypothesis Agent: 生成研究方向...")
    hypo = generate_directions(chat, trajectory.summary(bt.symbol), symbol=bt.symbol)
    directions = hypo["directions"][:max_directions]
    if not directions:
        print("[FAIL] 方向解析失败")
        print(hypo["raw_response"][:500])
        return None
    for i, d in enumerate(directions):
        print(f"  [{i}] {d['name']} | {d['logic'][:60]}")

    # ---- Step 2: Factor Generation (with trajectory context) ----
    print("\n[2] Factor Generation...")
    factors = []
    for d in directions:
        trajectory.add_direction(d["name"], d["logic"], bt.symbol)
        ctx = trajectory.direction_context(d["name"], bt.symbol)
        f = gen_factor(chat, d, trajectory_context=ctx)
        status = "OK" if f["valid"] else f"FAIL ({f['violation_reason'][:40]})"
        print(f"  [{d['name']}] {status}, retries={f['retries']}")
        factors.append(f)
        if f["valid"]:
            neg = dict(f)
            neg["formula"] = f"-( {f['formula']} )"
            factors.append(neg)

    # ---- Step 3: IS Grid Search + Code Screening ----
    print("\n[3] IS Grid Search + Code Screening...")
    is_entries = {}
    for i, f in enumerate(factors):
        if not f["valid"]:
            continue
        try:
            result = bt.evaluate(f["formula"], f["category"])
        except Exception as e:
            print(f"  [{i}] {f['direction']['name']}: IS FAILED ({e})")
            continue
        isr = result["is_result"]

        # Sharpe 全负 -> 记录教训
        if isr["sharpe_max"] < 0:
            trajectory.add_attempt(
                f["direction"]["name"],
                bt.symbol,
                f["formula"],
                isr,
                {"params": [], "pass_rate": "0/0"},
                "该方向IS Sharpe为负，方向本身可能错误",
            )
            trajectory.update_status(f["direction"]["name"], "failed", bt.symbol)
            print(f"  [{i}] {f['direction']['name']}: Sharpe max={isr['sharpe_max']}<0 -> 记录教训")
            continue

        # 确定性筛选（替代LLM评审）
        screen = screen_factor(isr, cfg)
        print(f"  [{i}] {f['direction']['name']}: Sharpe max={isr['sharpe_max']}, "
              f"占比={isr['sharpe_positive_ratio']:.0%}, "
              f"入选{len(screen['selected_params'])}组, 通过={screen['passed']}")

        if not screen["passed"]:
            trajectory.add_attempt(
                f["direction"]["name"],
                bt.symbol,
                f["formula"],
                isr,
                {"params": screen["selected_params"], "pass_rate": "0/0"},
                screen["reason"],
            )
            trajectory.update_status(f["direction"]["name"], "failed", bt.symbol)
            print(f"    -> 记录教训: {screen['reason']}")
            continue

        stats = screen["selected_stats"]
        print(f"    -> 粗糙度={stats['roughness']['combined']}, "
              f"Sharpe范围={stats['sharpe_range']}")
        is_entries[i] = {
            "factor": f,
            "result": result,
            "selected_params": screen["selected_params"],
        }

    if not is_entries:
        print("[FAIL] 没有因子通过IS筛选")
        return None

    # ---- Step 4: OOS Test + trajectory update ----
    print(f"\n[4] OOS Test ({len(is_entries)} factors)...")
    oos_outcomes = []
    for i, entry in is_entries.items():
        f = entry["factor"]
        params = entry["selected_params"]
        is_result = entry["result"]
        try:
            oos = bt.evaluate_oos(f["formula"], params, is_result["is_result"])
        except Exception as e:
            print(f"  [{i}] {f['direction']['name']}: OOS FAILED ({e})")
            continue

        factor_info = f"{f['direction']['name']} | {f['formula'][:80]}"
        verdict = ask_oos_judge(chat, oos["oos_report"], factor_info)

        # OOS pass rate among selected params
        oos_lookup = {(r["window"], r["threshold"]): r
                      for r in oos["oos_result"]["results"]}
        pass_count = sum(1 for w, th in params
                         if oos_lookup.get((w, th), {}).get("sharpe", 0) > 0)

        # OOS 达标参数 (Sharpe >= oos_sharpe_min)，记录并保存
        qualified = [(w, th, oos_lookup[(w, th)]["sharpe"])
                     for w, th in params
                     if oos_lookup.get((w, th), {}).get("sharpe", 0)
                     >= cfg["oos_sharpe_min"]]
        print(f"  [{i}] OOS达标参数: {len(qualified)}/{len(params)}个 "
              f"(Sharpe >= {cfg['oos_sharpe_min']})")
        for w, th, s in qualified:
            print(f"      ({w}, {th}) Sharpe={s:.3f}")

        oos_summary = {"params": params,
                       "pass_rate": f"{pass_count}/{len(params)}",
                       "oos_qualified_params": [(w, th) for w, th, _ in qualified],
                       "oos_qualified_count": f"{len(qualified)}/{len(params)}"}

        # 学习记录: 0个达标时加提示
        learning = verdict["learning"]
        if len(qualified) == 0:
            learning = f"0/{len(params)}达标，OOS全面失效。{learning}"

        trajectory.add_attempt(
            f["direction"]["name"],
            bt.symbol,
            f["formula"],
            is_result["is_result"],
            oos_summary,
            learning,
        )
        trajectory.update_status(
            f["direction"]["name"],
            "passed" if verdict["passed"] else "failed",
            bt.symbol,
        )

        oos_outcomes.append({
            "fidx": i,
            "params": params,
            "oos": oos,
            "verdict": verdict,
        })

        print(f"  [{i}] {f['direction']['name']}: {verdict['verdict']}")
        print(f"    分析: {verdict['analysis'][:120]}")
        print(f"    教训: {verdict['learning'][:120]}")

    print("\n" + "=" * 70)
    print("BATCH DONE")
    return {
        "directions": directions,
        "factors": factors,
        "is_entries": is_entries,
        "oos_outcomes": oos_outcomes,
    }


print("one_batch() ready.")

one_batch() ready.


## 运行一个 Batch

In [ ]:
batch = one_batch(max_directions=6)

BATCH START
[筛选配置] {'is_sharpe_min': 1.3, 'is_positive_ratio_min': 0.8, 'selected_roughness_max': 0.15, 'selected_trades_min': 50, 'selected_trades_max': 5000, 'min_selected_params': 3, 'oos_sharpe_min': 1.3}

[1] Hypothesis Agent: 生成研究方向...
  [0] 时序动量增强与衰竭 | 通过价格方向持续性和加速度变化识别趋势增强或衰竭阶段，捕捉顺势入场与离场时机。
  [1] 短期均值回复反转 | 当价格在单位时间内偏离近期均线或统计波动带过远时，押注其向均值回归。
  [2] 波动率聚集与状态切换 | 利用已实现波动率的自相关与状态转移，在高低波动聚集切换时调整信号触发条件。
  [3] 量价背离与动能确认 | 当价格创区间新高或新低而成交量未同步放大时，视为趋势动能衰减的背离信号。
  [4] 锚定效应：整数关口与历史高低点 | 价格首次逼近整数关口、前高前低等显著锚点时，关注突破延续或受阻反转的择时倾向。
  [5] 处置效应：持仓成本锚与解套压力 | 以近期量价分布估算平均持仓成本，当价格处于成本密集区附近时，解套或止损行为形成压力与支撑。

[2] Factor Generation...


## 批量运行 n 个 Batch

改 `N_BATCHES` / `MAX_DIRECTIONS` 后运行本 cell：连续跑 n 个 batch（带进度条），单个 batch 失败自动继续。

In [5]:
# ==================== 参数（改这里） ====================
N_BATCHES = 20
MAX_DIRECTIONS = 10
# ========================================================

import time

try:
    from tqdm import tqdm
    iterator = tqdm(range(N_BATCHES), desc="Batches", unit="batch")
except ImportError:
    iterator = range(N_BATCHES)

ok = fail = 0
t0 = time.time()
for n in iterator:
    try:
        r = one_batch(max_directions=MAX_DIRECTIONS)
        if r is None:
            fail += 1
            print(f"[WARN] Batch {n + 1}/{N_BATCHES} 无产出，继续")
        else:
            ok += 1
    except KeyboardInterrupt:
        print("\n[STOP] 用户中断")
        break
    except Exception as e:
        fail += 1
        print(f"[ERROR] Batch {n + 1}/{N_BATCHES} 崩溃: {e}，继续")

print("=" * 70)
print(f"DONE: ok={ok}, failed={fail}, 耗时 {(time.time() - t0) / 60:.1f} min")
print("=" * 70)
print(trajectory.summary(bt.symbol))

Batches:   0%|          | 0/20 [00:00<?, ?batch/s]

BATCH START
[筛选配置] {'is_sharpe_min': 1.3, 'is_positive_ratio_min': 0.8, 'selected_roughness_max': 0.15, 'selected_trades_min': 50, 'selected_trades_max': 5000, 'min_selected_params': 3, 'oos_sharpe_min': 1.3}

[1] Hypothesis Agent: 生成研究方向...
[retry] ReadTimeout -> sleep 2s (attempt 1/5)
  [0] 时序动量与趋势持续性 | 用价格相对过去N期收益或均线斜率识别趋势方向，并以波动率调整后的动量强度作为开仓条件。
  [1] 区间均值回复 | 在低波动震荡格局中，以价格偏离滚动均线或布林带的极端程度触发反向信号，回归价格中枢。
  [2] 波动率聚集与扩张启动 | 识别ATR或收益平方的波动率聚集特征，在低波压缩后波动率扩张初期参与方向性突破。
  [3] 量价背离与趋势衰竭 | 当价格创滚动新高/新低但成交量或OBV未同步确认甚至背离时，视为趋势衰竭的反转信号。
  [4] 锚定效应与关键锚点反应 | 以近期高低点、开盘价和整数关口为行为锚点，观察价格接近锚点时的受阻或突破反应。
  [5] 微观结构买卖压力 | 用K线实体位置、上下影线比例与成交量加权构造买卖压力代理，预测短期方向倾向。
  [6] 处置效应与成本密集区 | 以滚动VWAP或成交量密集区作为市场平均成本锚，价格接近该区域时解套或止盈压力形成反向或突破确认。

[2] Factor Generation...
  [时序动量与趋势持续性] OK, retries=0


c:\Users\user\Desktop\Quant\strategy\crypto_CTA\LLM_factor_mining\.\backtest\factor_lib.py:152: RuntimeWarning: invalid value encountered in divide
  np.asarray(x, dtype=np.float64) / np.asarray(y, dtype=np.float64),


  [区间均值回复] OK, retries=0
  [波动率聚集与扩张启动] OK, retries=0
  [量价背离与趋势衰竭] OK, retries=0
  [锚定效应与关键锚点反应] OK, retries=0
  [微观结构买卖压力] OK, retries=0
  [处置效应与成本密集区] OK, retries=0

[3] IS Grid Search + Code Screening...
  [0] 时序动量与趋势持续性: Sharpe max=0.7911, 占比=91%, 入选0组, 通过=False
    -> 记录教训: IS Sharpe最大值0.7911未达1.3
  [1] 时序动量与趋势持续性: Sharpe max=-0.234<0 -> 记录教训
  [2] 区间均值回复: Sharpe max=-0.4899<0 -> 记录教训
  [3] 区间均值回复: Sharpe max=0.755, 占比=71%, 入选0组, 通过=False
    -> 记录教训: IS Sharpe最大值0.755未达1.3
  [4] 波动率聚集与扩张启动: Sharpe max=0.4885, 占比=68%, 入选0组, 通过=False
    -> 记录教训: IS Sharpe最大值0.4885未达1.3
  [5] 波动率聚集与扩张启动: Sharpe max=-0.4587<0 -> 记录教训
  [6] 量价背离与趋势衰竭: Sharpe max=0.626, 占比=44%, 入选0组, 通过=False
    -> 记录教训: IS Sharpe最大值0.626未达1.3
  [7] 量价背离与趋势衰竭: Sharpe max=-0.6657<0 -> 记录教训
  [8] 锚定效应与关键锚点反应: Sharpe max=0.5544, 占比=26%, 入选0组, 通过=False
    -> 记录教训: IS Sharpe最大值0.5544未达1.3
  [9] 锚定效应与关键锚点反应: Sharpe max=0.1888, 占比=11%, 入选0组, 通过=False
    -> 记录教训: IS Sharpe最大值0.1888未达1.3
  [10] 微观结构买卖压力: Sharpe max=0.9294,

Batches:   5%|▌         | 1/20 [18:43<5:55:52, 1123.80s/batch]

...

  [13] 价格动量加速度与量能同步: Sharpe max=0.0588, 占比=2%, 入选0组, 通过=False
    -> 记录教训: IS Sharpe最大值0.0588未达1.3
[FAIL] 没有因子通过IS筛选
[WARN] Batch 18/20 无产出，继续
BATCH START
[筛选配置] {'is_sharpe_min': 1.3, 'is_positive_ratio_min': 0.8, 'selected_roughness_max': 0.15, 'selected_trades_min': 50, 'selected_trades_max': 5000, 'min_selected_params': 3, 'oos_sharpe_min': 1.3}

[1] Hypothesis Agent: 生成研究方向...
  [0] 边际买卖压力转折 | 用收盘位置与成交量构造逐K买卖压力的一阶增量，价格创新高或新低而边际压力转弱时反转，边际压力同步扩张时顺势。
  [1] 波动率冲击半衰期 | 用已实现波动率或高低范围的自相关衰减速度衡量波动冲击持续性，半衰期长时顺势，半衰期短时反转。
  [2] 风险调整动量 | 用收益除以同期ATR或已实现波动率构造单位风险收益，以滚动风险调整动量的方向和极端水平做顺势或反转切换。
  [3] 量能效率衰减 | 用价格绝对变化与成交量之比衡量单位价格移动所消耗的成交，趋势推进中边际量能效率持续下降但价格仍创新高或新低时反做，效率回
  [4] 路径效率与趋势质量 | 用净价格变化与累计路径长度之比衡量趋势效率，路径效率从低位回升时做趋势突破，效率极端高位后回落时做反转。
  [5] 极端收益后的量能确认 | 单K或短窗出现极端收益时，若量能未同步放大或收盘位于区间弱势侧则视为情绪过度并反向，量能与收盘位置确认则顺势。
  [6] 关键转折点锚定 | 以过去N期滚动最高价或最低价等显著转折点为锚，价格逼近锚点时缩量滞涨或滞跌做反转，放量有效突破则顺势。

[2] Factor Generation...
  [边际买卖压力转折] FAIL (), retries=2
  [波动率冲击半衰期] OK, retries=0
  [风险调整动量] OK, retries=0
  [量能效率衰减] 

Batches:  95%|█████████▌| 19/20 [6:42:03<27:04, 1624.55s/batch]

  [12] 关键转折点锚定: Sharpe max=0.034, 占比=1%, 入选0组, 通过=False
    -> 记录教训: IS Sharpe最大值0.034未达1.3
[FAIL] 没有因子通过IS筛选
[WARN] Batch 19/20 无产出，继续
BATCH START
[筛选配置] {'is_sharpe_min': 1.3, 'is_positive_ratio_min': 0.8, 'selected_roughness_max': 0.15, 'selected_trades_min': 50, 'selected_trades_max': 5000, 'min_selected_params': 3, 'oos_sharpe_min': 1.3}

[1] Hypothesis Agent: 生成研究方向...
  [0] 波动率压缩区间突破 | 用滚动波动率或布林带宽分位识别低波动压缩，价格突破该压缩区间上/下沿后顺势，回归区间内不做。
  [1] 均线锚定过度偏离反转 | 以滚动均线为锚，当价格经ATR标准化后的偏离达到极端时反向交易至均线回归，不追单边延伸。
  [2] 滚动成交量分布高低量节点 | 用滚动价格区间内成交量分布识别高量争夺区与低量真空区，价格进入低量区快速顺势穿越，进入高量区等待突破或回归。
  [3] 收益方向一致性与趋势质量 | 用滚动窗口内同号收益占比或收益序列自相关衡量趋势一致性，一致性强且波动可控时顺势，一致性骤降时退出或反手。
  [4] 窄幅高量吸收后的方向突破 | 在价格波动收窄且成交量异常放大但价格未突破的状态视为多空换手吸收，以随后突破该窄幅区间的方向做有效顺势。
  [5] 收益分布尾部极端后的反转 | 滚动收益偏度或峰度进入极端区域表示单边极端K线集中释放，后续倾向均值回复，分布形态正常化后停止反向交易。

[2] Factor Generation...
  [波动率压缩区间突破] OK, retries=0
  [均线锚定过度偏离反转] OK, retries=0
  [滚动成交量分布高低量节点] OK, retries=0
  [收益方向一致性与趋势质量] OK, retries=2
  [窄幅高量吸收后的方向突破] OK, retries=1
  [收益分布尾部极端后的反转] O

Batches: 100%|██████████| 20/20 [6:58:42<00:00, 1256.14s/batch]

  [4] OOS达标参数: 0/21个 (Sharpe >= 1.3)
  [4] 滚动成交量分布高低量节点: 失败
    分析: OOS通过率0/21=0%，且IS→OOS Sharpe中位数保持率约-107%，远低于40%门槛；虽然粗糙度仅恶化1.8x，但所有IS优选参数在OOS全面转负，说明不是单纯表面锯齿化，而是因子逻辑在样本外失效，属于典型IS过拟合。
    教训: 该方向不应继续在单标的、单周期上优化参数，应降低参数敏感度并加入多市场/多时段验证，或采用稳定区域参数选择与市场状态过滤后再测试。

BATCH DONE
DONE: ok=3, failed=17, 耗时 418.7 min
方向「收益分布尾部极端后的反转」[SANDUSDT_1H][failed] 尝试2次: 滚动收益偏度或峰度进入极端区域表示单边极端K线集中释放，后续倾向均值回复，分布形态正常化后停止反向交易。
  - IS Sharpe=0.8447, 粗糙度=0.2261, OOS={'params': [], 'pass_rate': '0/0'}
    教训: IS Sharpe最大值0.8447未达1.3
  - IS Sharpe=0.5153, 粗糙度=0.2277, OOS={'params': [], 'pass_rate': '0/0'}
    教训: IS Sharpe最大值0.5153未达1.3
方向「窄幅高量吸收后的方向突破」[SANDUSDT_1H][failed] 尝试2次: 在价格波动收窄且成交量异常放大但价格未突破的状态视为多空换手吸收，以随后突破该窄幅区间的方向做有效顺势。
  - IS Sharpe=0.7871, 粗糙度=0.1848, OOS={'params': [], 'pass_rate': '0/0'}
    教训: IS Sharpe最大值0.7871未达1.3
  - IS Sharpe=-0.4512, 粗糙度=0.1593, OOS={'params': [], 'pass_rate': '0/0'}
    教训: 该方向IS Sharpe为负，方向本身可能错误
方向「收益方向一致性与趋势质量」[SANDUSDT_1H][failed] 尝试2次: 用滚动窗口内同号收益占比或收益序列自相关衡量趋势一致性，一致性强且波

## 查看轨迹

In [ ]:
# 查看研究轨迹（下个 batch 会喂给 Hypothesis Agent）
print(trajectory.summary(bt.symbol))

方向「时序动量持续性」[SANDUSDT_1H][failed] 尝试1次: 利用过去多时间尺度的收益符号、强度与路径平滑度，捕捉趋势延续概率较高的阶段。
  - IS Sharpe=0.9491, 粗糙度=0.2144, OOS={'params': [], 'pass_rate': '0/0'}
    教训: IS Sharpe最大值0.9491未达1.3
方向「波动率标准化短期反转」[SANDUSDT_1H][failed] 尝试1次: 当价格对短期均线或近端价格中枢的偏离经已实现波动率标准化后达到极端时，押注向中枢回归。
  - IS Sharpe=-0.78, 粗糙度=0.1456, OOS={'params': [], 'pass_rate': '0/0'}
    教训: 该方向IS Sharpe为负，方向本身可能错误
方向「波动率聚集与压缩突破」[SANDUSDT_1H][failed] 尝试1次: 识别已实现波动率的持续性以及低波动压缩状态，作为后续趋势方向选择或状态切换的触发器。
  - IS Sharpe=0.8424, 粗糙度=0.2196, OOS={'params': [], 'pass_rate': '0/0'}
    教训: IS Sharpe最大值0.8424未达1.3
方向「量价背离与量能确认」[SANDUSDT_1H][failed] 尝试1次: 当价格创近N期新高/新低而成交量或能量潮未同步放大时，对趋势动能弱化做反向预警。
  - IS Sharpe=0.8578, 粗糙度=0.304, OOS={'params': [], 'pass_rate': '0/0'}
    教训: IS Sharpe最大值0.8578未达1.3
方向「行为金融锚定水平反应」[SANDUSDT_1H][failed] 尝试1次: 以前期高低点、成交密集区或整数关口作为锚点，观察价格接近或突破锚点后的企稳与假突破行为。
  - IS Sharpe=0.7174, 粗糙度=0.2337, OOS={'params': [], 'pass_rate': '0/0'}
    教训: IS Sharpe最大值0.7174未达1.3
方向「处置效应与成本密集抛压」[SANDUSDT_1H][failed] 尝试1次: 用成交量加权的近端成本区代理盈亏